In [0]:
SELECT * FROM workspace.default.final_quick_comm_dataset LIMIT 10

In [0]:
DESC workspace.default.final_quick_comm_dataset

In [0]:
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value),2) AS GMV,
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id),2) AS AOV,
    ROUND(AVG(review_score),2) AS avg_review_score
FROM workspace.default.final_quick_comm_dataset;

In [0]:
WITH customer_orders AS (

    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id) AS order_count

    FROM workspace.default.final_quick_comm_dataset

    GROUP BY 1
)

SELECT
    CASE
        WHEN order_count = 1 THEN 'One-Time'
        ELSE 'Repeat'
    END AS customer_type,

    COUNT(*) AS customers

FROM customer_orders

GROUP BY 1;

In [0]:
WITH customer_orders AS (

    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM workspace.default.final_quick_comm_dataset
    GROUP BY 1
),

repeat_customers AS (

    SELECT
        customer_unique_id

    FROM customer_orders
    WHERE total_orders > 1
),

order_level AS (

    SELECT
        order_id,
        customer_unique_id,
        product_category_name_english,
        MAX(payment_value) AS order_value

    FROM workspace.default.final_quick_comm_dataset

    GROUP BY 1,2,3
)

SELECT

    product_category_name_english,
    ROUND(SUM(order_value),2) AS total_revenue,
    ROUND(
        SUM(
            CASE
                WHEN customer_unique_id IN (
                    SELECT customer_unique_id
                    FROM repeat_customers
                )
                THEN order_value
                ELSE 0
            END
        ),
    2) AS repeat_customer_revenue,

    ROUND(
        SUM(
            CASE
                WHEN customer_unique_id IN (
                    SELECT customer_unique_id
                    FROM repeat_customers
                )
                THEN order_value
                ELSE 0
            END
        ) * 100.0
        /
        SUM(order_value),
    2) AS repeat_purchase_contribution_pct

FROM order_level

GROUP BY 1

ORDER BY repeat_purchase_contribution_pct DESC;

In [0]:
WITH first_purchase AS (

    SELECT
        customer_unique_id,
        date(MIN(date_trunc('month', order_purchase_timestamp))) AS cohort_month

    FROM workspace.default.final_quick_comm_dataset

    GROUP BY 1
),

monthly_orders AS (

    SELECT
        customer_unique_id,
        date(date_trunc('month', order_purchase_timestamp)) AS order_month

    FROM workspace.default.final_quick_comm_dataset
)

SELECT
    fp.cohort_month,
    mo.order_month,
    COUNT(DISTINCT mo.customer_unique_id) AS active_customers

FROM first_purchase fp

JOIN monthly_orders mo
ON fp.customer_unique_id = mo.customer_unique_id

GROUP BY 1,2
ORDER BY 1,2;

In [0]:
SELECT

    CASE
        WHEN order_delivered_customer_date >
             order_estimated_delivery_date

        THEN 'Delayed'

        ELSE 'On-Time'
    END AS delivery_status,

    COUNT(DISTINCT order_id) AS orders,

    ROUND(AVG(review_score),2) AS avg_review_score,

    ROUND(AVG(payment_value),2) AS avg_order_value

FROM workspace.default.final_quick_comm_dataset

WHERE order_status = 'delivered'

GROUP BY 1;

In [0]:
SELECT

    product_category_name_english,

    COUNT(DISTINCT order_id) AS total_orders,

    ROUND(SUM(payment_value),2) AS revenue,

    ROUND(AVG(review_score),2) AS avg_review_score,

    ROUND(AVG(delivery_delay_days),2) AS avg_delay_days

FROM workspace.default.final_quick_comm_dataset

GROUP BY 1

ORDER BY revenue DESC
LIMIT 10;

In [0]:
SELECT

    product_category_name_english,

    COUNT(DISTINCT order_id) AS cancelled_orders

FROM workspace.default.final_quick_comm_dataset

WHERE order_status = 'canceled'
and product_category_name_english IS NOT NULL
GROUP BY 1
ORDER BY cancelled_orders DESC
LIMIT 10;

In [0]:
SELECT

    order_status,

    COUNT(DISTINCT order_id) AS orders

FROM workspace.default.final_quick_comm_dataset

GROUP BY 1

ORDER BY orders DESC;

In [0]:
WITH order_level AS (
    SELECT
        customer_unique_id,
        order_id,
        MAX(payment_value) AS order_value,
        MAX(review_score) AS review_score,
        CASE
            WHEN MAX(order_delivered_customer_date) > MAX(order_estimated_delivery_date)
            THEN 'Delayed'
            ELSE 'On-Time'
        END AS delivery_status
    FROM workspace.default.final_quick_comm_dataset
    WHERE order_status = 'delivered'
    GROUP BY customer_unique_id, order_id
),

customer_level AS (
    SELECT
        customer_unique_id,
        CASE
            WHEN COUNT(DISTINCT order_id) = 1
            THEN 'One-time'
            ELSE 'Repeat'
        END AS buyer_type,
        ROUND(AVG(order_value),2) AS avg_spend_per_order
    FROM order_level
    GROUP BY customer_unique_id
)

SELECT
    c.buyer_type,
    o.delivery_status,
    COUNT(DISTINCT o.customer_unique_id) AS customers,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND((COUNT(DISTINCT o.order_id)/COUNT(DISTINCT o.customer_unique_id)),2) as orders_per_customer,
    ROUND(AVG(o.order_value),2) AS avg_order_value,
    ROUND(AVG(o.review_score),2) AS avg_review_score,
    ROUND(AVG(c.avg_spend_per_order),2) AS avg_spend_per_order
FROM order_level o
JOIN customer_level c
ON o.customer_unique_id = c.customer_unique_id
GROUP BY 1,2
ORDER BY 1,2;

In [0]:
WITH customer_orders AS (

    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id) AS total_orders,

        MAX(
            CASE
                WHEN order_delivered_customer_date >
                     order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) AS had_delay

    FROM workspace.default.final_quick_comm_dataset

    WHERE order_status = 'delivered'

    GROUP BY 1
)

SELECT

    CASE
        WHEN had_delay = 1 THEN 'Delayed Experience'
        ELSE 'On-Time Experience'
    END AS delivery_experience,

    COUNT(*) AS customers,

    ROUND(
        AVG(
            CASE
                WHEN total_orders > 1 THEN 1
                ELSE 0
            END
        ) * 100,
    2) AS repeat_purchase_rate

FROM customer_orders

GROUP BY 1;

In [0]:
WITH order_level AS (

    SELECT
        customer_unique_id,
        order_id,
        product_category_name_english,

        MAX(payment_value) AS order_value,

        MAX(review_score) AS review_score,

        CASE
            WHEN MAX(order_delivered_customer_date) >
                 MAX(order_estimated_delivery_date)

            THEN 'Delayed'

            ELSE 'On-Time'
        END AS delivery_status

    FROM workspace.default.final_quick_comm_dataset

    WHERE order_status = 'delivered'

    GROUP BY
        customer_unique_id,
        order_id,
        product_category_name_english
),

customer_type AS (

    SELECT
        customer_unique_id,

        CASE
            WHEN COUNT(DISTINCT order_id) = 1
            THEN 'One-time'

            ELSE 'Repeat'
        END AS buyer_type

    FROM order_level

    GROUP BY 1
)

SELECT

    ct.buyer_type,

    ol.delivery_status,
    concat(ct.buyer_type,' - ',ol.delivery_status) as purchase_type,

    ol.product_category_name_english,

    COUNT(DISTINCT ol.order_id) AS orders,

    ROUND(AVG(ol.order_value),2) AS avg_order_value,

    ROUND(AVG(ol.review_score),2) AS avg_review_score

FROM order_level ol

JOIN customer_type ct
ON ol.customer_unique_id = ct.customer_unique_id

GROUP BY 1,2,3,4

HAVING COUNT(DISTINCT ol.order_id) > 100

ORDER BY avg_order_value DESC;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.